# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is a Croissant schema available at:
[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset @id: {metadata.id}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

This helps us understand the data's structure and select the components to extract and analyze.

In [ ]:
# List available record sets and their field IDs
record_sets = list(dataset.record_sets)

print("Available record sets:")
for rs in record_sets:
    print(f"- @id: {rs.id} | name: {rs.name}")
    if rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    - @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    print("")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Let's extract data from each available record set
dataframes = {}

# We'll collect each record set's @id for reference
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:  # If any records loaded
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records. Fields: {df.columns.tolist()}")
    else:
        print("  No records found.")
print("\nDataFrames loaded for these record sets:")
print(list(dataframes.keys()))


## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll demonstrate these steps on the largest non-empty record set.

In [ ]:
# Select the largest non-empty record set for demonstration
if dataframes:
    main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k]))
    main_df = dataframes[main_record_set_id]
    print(f"Using record set: {main_record_set_id} with {len(main_df)} records.")
    print("Columns available:", main_df.columns.tolist())
else:
    print("No non-empty dataframes found.")

In [ ]:
# Identify a numeric field for demonstration
import numpy as np

# Attempt to find a numeric field
numeric_candidates = []
for col in main_df.columns:
    # Try converting column to numeric to check
    try:
        pd.to_numeric(main_df[col].dropna(), errors='raise')
        numeric_candidates.append(col)
    except Exception:
        continue

print(f"Numeric field candidates: {numeric_candidates}")

if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Selecting field '{numeric_field}' for filtering and normalization.")
    
    # Convert to numeric
    main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')
    threshold = main_df[numeric_field].mean()
    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"Filtered records in '{main_record_set_id}' with {numeric_field} > mean:")
    print(filtered_df.head())
    
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Try grouping on the first non-numeric field
    non_numeric_group = None
    for col in main_df.columns:
        if col != numeric_field and main_df[col].dtype == object:
            non_numeric_group = col
            break
    if non_numeric_group:
        grouped = filtered_df.groupby(non_numeric_group)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field}' by '{non_numeric_group}':")
        print(grouped.head())
else:
    print("No obvious numeric field found for demonstration.")

## 5. Visualization

Visualize distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if numeric_candidates:
    # Histogram of the selected numeric field
    plt.figure(figsize=(8,4))
    main_df[numeric_field].hist(bins=30)
    plt.xlabel(numeric_field)
    plt.title(f'Distribution of {numeric_field}')
    plt.show()
    
    # If a grouping field was found, display a bar plot
    if non_numeric_group:
        grouped.set_index(non_numeric_group)[numeric_field].plot(kind='bar', figsize=(8,4))
        plt.ylabel(f"Mean {numeric_field}")
        plt.title(f"Grouped mean of {numeric_field} by {non_numeric_group}")
        plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR² dataset with `mlcroissant`, explored its available record sets and fields using their `@id`s, extracted data into Pandas DataFrames, performed basic numeric filtering and normalization, and visualized field distributions. You can now extend this workflow for deeper analysis or custom data transformations pertinent to your domain questions.

*Remember: Always refer to fields and record sets using their `@id` for full reproducibility and clarity when working with Croissant-based datasets.*